# HOG + SVM — Random-Image Split Experiment

Compare the random-image split with the driver-based split
to investigate potential performance inflation.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path



In [2]:
# Load random-split metadata
random_train = pd.read_csv("../data/splits/random/train.csv")
random_val = pd.read_csv("../data/splits/random/val.csv")

# Locate the original images
sample_img = random_train.iloc[0]["img"]
sample_path = next(Path("../data/raw").rglob(sample_img))
image_root = sample_path.parent.parent

# Add image paths to both datasets
for data in [random_train, random_val]:
    data["filepath"] = data.apply(
        lambda row: str(
            image_root / row["classname"] / row["img"]
        ),
        axis=1
    )

# Verify datasets
print("Training:", random_train.shape)
print("Validation:", random_val.shape)
print("First image exists:",
      Path(random_train.iloc[0]["filepath"]).is_file())

Training: (15696, 5)
Validation: (3364, 5)
First image exists: True


## Step 2 — Extract HOG Features
Convert training and validation images into numerical HOG features
using the same configuration as the driver-based experiment.

In [3]:
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize
from skimage.feature import hog
from tqdm import tqdm

# Convert one image into HOG features
def extract_hog(image_path):
    image = imread(image_path)
    gray = rgb2gray(image)
    gray = resize(gray, (64, 64), anti_aliasing=True)

    features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    return features


# Convert all images into numerical features and labels
def build_features(data):
    X = np.array([
        extract_hog(path)
        for path in tqdm(data["filepath"])
    ])

    y = data["classname"].to_numpy(dtype="U10")

    return X, y


# Extract training features
X_train, y_train = build_features(random_train)

# Extract validation features
X_val, y_val = build_features(random_val)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

100%|██████████| 3364/3364 [00:27<00:00, 121.68it/s]

Training: (15696, 1764)
Validation: (3364, 1764)


## Step 3 — Save Random-Split HOG Features
Save training and validation features and labels as NumPy files for reuse.

In [4]:
# Create a folder for random-split HOG features
feature_dir = Path("../data/processed/hog/random")
feature_dir.mkdir(parents=True, exist_ok=True)

# Save training features and labels
np.save(feature_dir / "X_train.npy", X_train)
np.save(feature_dir / "y_train.npy", y_train)

# Save validation features and labels
np.save(feature_dir / "X_val.npy", X_val)
np.save(feature_dir / "y_val.npy", y_val)

print("Random-split HOG features saved successfully!")

Random-split HOG features saved successfully!


## Step 4 — Train HOG + SVM

Train a Linear SVM using the random-image split and the same
hyperparameters as the driver-based experiment.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
import numpy as np

# Load previously saved HOG features
feature_dir = Path("../data/processed/hog/random")

X_train = np.load(feature_dir / "X_train.npy")
y_train = np.load(feature_dir / "y_train.npy")

X_val = np.load(feature_dir / "X_val.npy")
y_val = np.load(feature_dir / "y_val.npy")

# Create the same pipeline as our first experiment
pipeline_random = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", LinearSVC(
        C=1.0,
        max_iter=10000,
        random_state=42
    ))
])

# Measure training time
import time

start = time.perf_counter()

pipeline_random.fit(X_train, y_train)

training_seconds = time.perf_counter() - start

print("SVM training completed!")
print(f"Training time: {training_seconds:.2f} seconds")

SVM training completed!
Training time: 1294.91 seconds


## Step 5 — Save Random-Split SVM Model

Save the fitted pipeline for future evaluation without retraining.

In [6]:
import joblib
from pathlib import Path

# Create the models folder
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save the trained pipeline
joblib.dump(
    pipeline_random,
    model_dir / "hog_svm_random.joblib"
)

print("Random-split SVM saved successfully!")

Random-split SVM saved successfully!


In [7]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Predict validation labels
y_pred_random = pipeline_random.predict(X_val)

# Calculate performance
random_accuracy = accuracy_score(y_val, y_pred_random)
random_macro_f1 = f1_score(y_val, y_pred_random, average="macro")

print(f"Random-split Accuracy: {random_accuracy:.2%}")
print(f"Random-split Macro-F1: {random_macro_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred_random, zero_division=0))

Random-split Accuracy: 98.66%
Random-split Macro-F1: 0.9866

Classification Report:
              precision    recall  f1-score   support

          c0       0.98      0.97      0.98       374
          c1       0.99      1.00      1.00       340
          c2       0.99      0.99      0.99       348
          c3       0.98      0.98      0.98       352
          c4       0.98      0.98      0.98       349
          c5       1.00      1.00      1.00       347
          c6       0.99      1.00      1.00       348
          c7       0.99      1.00      1.00       300
          c8       0.97      0.98      0.98       286
          c9       0.98      0.97      0.97       320

    accuracy                           0.99      3364
   macro avg       0.99      0.99      0.99      3364
weighted avg       0.99      0.99      0.99      3364



## Step 7 — Compare Split Protocols

Compare HOG + SVM performance under random-image
and driver-disjoint validation.

In [8]:
# Compare both experiments
comparison = pd.DataFrame({
    "Split": ["Driver-based", "Random-image"],
    "Accuracy": [0.4252, random_accuracy],
    "Macro-F1": [0.4180, random_macro_f1]
})

# Calculate accuracy difference
gap = (random_accuracy - 0.4252) * 100

print(f"Accuracy gap: {gap:.2f} percentage points")

display(comparison)

# Save comparison for the report
output_dir = Path("../results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "hog_svm_split_comparison.csv",
    index=False
)

Accuracy gap: 56.14 percentage points


,Split,Accuracy,Macro-F1
0,Driver-based,0.425200,0.418000
1,Random-image,0.986623,0.986551


## Step 8 — Save Random-Split Evaluation Results

Save metrics and predictions for reproducibility and comparison.

In [9]:
import json
from sklearn.metrics import classification_report

# Create results folder
results_dir = Path("../results/hog_svm/random")
results_dir.mkdir(parents=True, exist_ok=True)

# Save model performance
metrics = {
    "model": "HOG + LinearSVC",
    "split": "random",
    "accuracy": float(random_accuracy),
    "macro_f1": float(random_macro_f1),
    "training_time_seconds": training_seconds
}

with open(results_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

# Save per-class metrics
report = classification_report(
    y_val, y_pred_random,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).T.to_csv(
    results_dir / "classification_report.csv"
)

# Save individual predictions
predictions = random_val.copy()
predictions["actual"] = y_val
predictions["predicted"] = y_pred_random

predictions.drop(
    columns=["filepath"], errors="ignore"
).to_csv(results_dir / "predictions.csv", index=False)

print("Random-split results saved!")

Random-split results saved!
